In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import sys,subprocess
URL='https://github.com/RICHAAARC/SC-SSTW.git'; BRANCH='c2a-2a-colab-preparation'; SOURCE=Path('/content/c2t1_prevae_repair_source')
if SOURCE.exists():
    if subprocess.check_output(['git','-C',str(SOURCE),'remote','get-url','origin'],text=True).strip()!=URL: raise RuntimeError('unexpected origin')
else:
    subprocess.run(['git','init',str(SOURCE)],check=True); subprocess.run(['git','-C',str(SOURCE),'remote','add','origin',URL],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','--depth','1','origin',BRANCH],check=True); subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','--force','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','diffusers','transformers','accelerate','ftfy','sentencepiece','safetensors','huggingface_hub','numpy','Pillow'],check=True); subprocess.run(['ffmpeg','-version'],check=True)


In [ ]:
from datetime import datetime,timezone
CONFIG=SOURCE/'runtime/c2t1/c2t1_prevae_repair_run.json'; RUN_ID='c2t1_prevae_repair_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); OUTPUT=Path('/content/drive/MyDrive/Video-WM/C2T1_PreVAERepair')/RUN_ID
if OUTPUT.exists(): raise FileExistsError(OUTPUT)
import os,signal
cmd=[sys.executable,'-u','-m','runtime.c2t1.run_prevae_repair','--config',str(CONFIG),'--output',str(OUTPUT)]; LOG=OUTPUT.parent/f'{RUN_ID}.launcher.log'; LOG.parent.mkdir(parents=True,exist_ok=True)
with LOG.open('w') as log:
    p=subprocess.Popen(cmd,cwd=SOURCE,start_new_session=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout: print(line,end=''); log.write(line); log.flush()
        code=p.wait()
    except BaseException:
        try: p.send_signal(signal.SIGTERM)
        except ProcessLookupError: pass
        try: p.wait(timeout=5)
        except subprocess.TimeoutExpired: os.killpg(p.pid,signal.SIGKILL); p.wait()
        raise
print((OUTPUT/'result.json').read_text() if (OUTPUT/'result.json').exists() else 'No result file')
if code: raise subprocess.CalledProcessError(code,cmd)
